# Importing Lib



In [ ]:

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv1D, BatchNormalization, Activation, Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import categorical_crossentropy
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
import pandas as pd

data = pd.read_csv(
    '/kaggle/input/datasets/luyucwnu/tlmuav-anomaly-detection-datasets/dataset/GPS/ALL_FAIL_LOG_GPS_0.csv'
)

print(data.shape)
print(data.columns.tolist())
data.head()

# Preprocessing and Exploring data



In [ ]:
print("Dataset Shape:", data.shape)
print("\nColumns:\n", data.columns)
print("\nSample Data:\n", data.head())
print("\nMissing Values:\n", data.isnull().sum())

In [ ]:
print("\nData Types:\n", data.dtypes)

In [ ]:
import matplotlib.pyplot as plt
if 'labels' in data.columns:
    label_counts = data['labels'].value_counts()
    # Define specific colors for 5 labels
    colors = ["#FF91A4", "#f6c344", "#79d970", "#F4E1C1", "#6495ED"]

    # Plot the donut chart
    plt.figure(figsize=(4, 5))
    plt.pie(label_counts, labels=label_counts.index, autopct='%1.1f%%', startangle=90, colors=colors, wedgeprops={'width': 0.4})
    plt.axis('equal')
    plt.show()
    



In [ ]:
# Encoding Categorical Data
if 'labels' in data.columns:
    label_encoder = LabelEncoder()
    data['labels'] = label_encoder.fit_transform(data['labels'])
    print("\nLabel Encoding Complete. Classes:", label_encoder.classes_)

In [ ]:
# Feature Selection
X = data.drop(columns=['labels']).values  # Features
y = data['labels'].values  # Target

In [ ]:
X.shape

In [ ]:
# Scaling Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# One-hot encode the labels
y_encoded = to_categorical(y, num_classes=len(np.unique(y)))

In [ ]:
window_size = 5

X_windows = []
y_windows = []

for i in range(len(X_scaled) - window_size + 1):
    X_windows.append(X_scaled[i:i+window_size])
    y_windows.append(y_encoded[i+window_size-1])   # <-- Correct

X_windows = np.array(X_windows)
y_windows = np.array(y_windows)

print(X_windows.shape)
print(y_windows.shape)

input_shape = X_windows.shape[1:]
num_classes = y_windows.shape[1]

In [ ]:
for label in data['labels'].unique():
    print(f"\nValues for label: {label}")
    print(data[data['labels'] == label].head())

In [ ]:
for label in data['labels'].unique():
    print(f"\nValues for label: {label}")
    print(data[data['labels'] == label].head())

# Model

In [ ]:
# Define the TCN Block
def TCN_Block(filters, kernel_size, dilation_rate):
    return tf.keras.Sequential([
        Conv1D(filters=filters, kernel_size=kernel_size, padding='causal', dilation_rate=dilation_rate),
        BatchNormalization(),
        Activation('relu')
    ])

In [ ]:
# Define the Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], input_shape[-1]), initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(shape=(input_shape[-1],), initializer="zeros", trainable=True)
        self.u = self.add_weight(shape=(input_shape[-1],), initializer="glorot_uniform", trainable=True)

    def call(self, inputs):
        u_t = tf.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        a_t = tf.nn.softmax(tf.tensordot(u_t, self.u, axes=1), axis=1)
        output = tf.reduce_sum(inputs * tf.expand_dims(a_t, -1), axis=1)
        return output

In [ ]:
# Define the TCN with Attention Model
def build_tcn_attention_model(input_shape, num_classes):
    # inputs = Input(shape=input_shape)
# def build_tcn_attention_model(X_reshaped, num_classes):
    inputs = Input(shape=input_shape)
     # Replacing input_shape with X_reshaped.shape
    # TCN Layers
    x = TCN_Block(filters=64, kernel_size=3, dilation_rate=4)(inputs)
    x = TCN_Block(filters=64, kernel_size=3, dilation_rate=8)(x)
    x = TCN_Block(filters=64, kernel_size=3, dilation_rate=16)(x)

    # Attention mechanism
    attention_output = AttentionLayer()(x)

    # Classification
    classification = Dense(num_classes, activation='softmax')(attention_output)

    # Define and compile the model
    model = Model(inputs=inputs, outputs=classification)
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss=categorical_crossentropy,
                  metrics=['accuracy'])

    return model


In [ ]:
# Input shape for TCN
input_shape = (X_windows.shape[1], X_windows.shape[2])
num_classes = y_windows.shape[1]

In [ ]:
# Perform k-fold cross-validation
#from tensorflow.keras.callbacks import EarlyStopping
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_no = 1
accuraciesKF = []
label_accuraciesKF = []



for train_index, val_index in kf.split(X_windows):
    print(f"\nTraining fold {fold_no}...")

    # Split data
    X_train_foldKF, X_val_foldKF = X_windows[train_index], X_windows[val_index]
    y_train_foldKF, y_val_foldKF = y_windows[train_index], y_windows[val_index]

    # Build and train the model
    # modelKF = build_tcn_attention_model(input_shape, num_classes)
    modelKF = build_tcn_attention_model(input_shape, num_classes)
   # modelKF.fit(X_train_foldKF, y_train_foldKF, epochs=50, batch_size=32, verbose=0, validation_data=(X_val_foldKF, y_val_foldKF))
    history = modelKF.fit(
    X_train_foldKF,
    y_train_foldKF,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_foldKF, y_val_foldKF),
    verbose=1
    )
    
    # Evaluate the model
    lossKF, accuracyKF = modelKF.evaluate(X_val_foldKF, y_val_foldKF, verbose=0)
    print(f"Validation Accuracy for fold {fold_no}: {accuracyKF:.4f}")
    accuraciesKF.append(accuracyKF)

    # Predictions
    y_predKF = modelKF.predict(X_val_foldKF)
    y_pred_classesKF = np.argmax(y_predKF, axis=1)
    y_true_classesKF = np.argmax(y_val_foldKF, axis=1)

    # Accuracy per label
    label_accKF = []
    for label in range(num_classes):
        label_indicesKF = (y_true_classesKF == label)
        label_accKF.append(accuracy_score(y_true_classesKF[label_indicesKF], y_pred_classesKF[label_indicesKF]))
    label_accuraciesKF.append(label_accKF)

    fold_no += 1



In [ ]:
modelKF.summary()

In [ ]:
# Calculate average accuracy
average_accuracyKF = np.mean(accuraciesKF)
print(f"\nAverage Accuracy across all folds: {average_accuracyKF:.4f}")

In [ ]:
# Average accuracy per label
label_accuraciesKF = np.mean(label_accuraciesKF, axis=0)
for idx, acc in enumerate(label_accuraciesKF):
    print(f"Label {idx} Accuracy: {acc:.4f}")

In [ ]:
# Evaluate the last fold predictions
y_predKF = modelKF.predict(X_val_foldKF)
y_pred_classesKF = np.argmax(y_predKF, axis=1)
y_true_classesKF = np.argmax(y_val_foldKF, axis=1)

In [ ]:
import pandas as pd
import numpy as np

feature_names = ['LineNo', 'TimeUS', 'I', 'Status', 'GMS', 'GWk', 'NSats', 'HDop', 'Lat',
       'Lng', 'Alt', 'Spd', 'GCrs', 'VZ', 'Yaw', 'U']
# Predict on all windows
y_pred_probs = modelKF.predict(X_windows, verbose=0)

# Convert probabilities to labels
y_pred_labels = np.argmax(y_pred_probs, axis=1)
y_true_labels = np.argmax(y_windows, axis=1)

# Find misclassified windows
mismatch_indices = np.where(y_true_labels != y_pred_labels)[0]

print("Total Misclassified Windows:", len(mismatch_indices))

# Extract the last time step from each misclassified window
X_last = X_windows[mismatch_indices, -1, :]

# Create DataFrame
df_mismatches = pd.DataFrame(
    X_last,
    columns=feature_names
)

# Add labels
df_mismatches["Actual_Label"] = y_true_labels[mismatch_indices]
df_mismatches["Predicted_Label"] = y_pred_labels[mismatch_indices]

# Add original window index
df_mismatches.insert(0, "Window_Index", mismatch_indices)

print(df_mismatches)

# Optional: Save to CSV
df_mismatches.to_csv("Misclassified_Windows.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np

feature_names = ['LineNo', 'TimeUS', 'I', 'Status', 'GMS', 'GWk', 'NSats', 'HDop', 'Lat',
       'Lng', 'Alt', 'Spd', 'GCrs', 'VZ', 'Yaw', 'U']
# Predict on windowed data
y_pred_probs = modelKF.predict(X_windows, verbose=0)

# Convert probabilities to labels
y_pred_labels = np.argmax(y_pred_probs, axis=1)
y_true_labels = np.argmax(y_windows, axis=1)

# Find misclassified windows
mismatch_indices = np.where(y_true_labels != y_pred_labels)[0]

# Extract the last time step of each misclassified window
X_last = X_windows[mismatch_indices, -1, :]

# Create dataframe
df_mismatches = pd.DataFrame(
    X_last,
    columns=feature_names
)

# Add metadata
df_mismatches.insert(0, "Window_Index", mismatch_indices)
df_mismatches["Actual_Label"] = y_true_labels[mismatch_indices]
df_mismatches["Predicted_Label"] = y_pred_labels[mismatch_indices]

print(df_mismatches)

In [ ]:
import pandas as pd
import numpy as np

# Step 1: Identify correctly classified windows
correct_indices = np.where(y_true_labels == y_pred_labels)[0]

# Step 2: Extract the last time step of each correctly classified window
X_last = X_windows[correct_indices, -1, :]

# Step 3: Create DataFrame
df_correct = pd.DataFrame(
    X_last,
    columns=feature_names
)

# Add metadata
df_correct.insert(0, "Window_Index", correct_indices)
df_correct["Actual_Label"] = y_true_labels[correct_indices]
df_correct["Predicted_Label"] = y_pred_labels[correct_indices]

# Step 4: Save to CSV
df_correct.to_csv("correctly_classified.csv", index=False)

# Display
print(df_correct.head())

In [ ]:
df_mismatches.to_csv("misclassified_samples.csv", index=False)
print("CSV file 'misclassified_samples.csv' saved successfully!")


In [ ]:
# Classification report
print("\nClassification Report:\n")
print(classification_report(y_true_classesKF, y_pred_classesKF,digits = 4))

In [ ]:
# Confusion Matrix
conf_matrixKF = tf.math.confusion_matrix(y_true_classesKF, y_pred_classesKF)
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrixKF, annot=True, fmt='d', cmap='YlOrRd')
plt.xlabel('Predicted')
plt.ylabel('True')

plt.title('Confusion Matrix for Last Fold')
plt.show()

In [ ]:
if 'labels' in data.columns:
    label_countsKF = data['labels'].value_counts()
    print("\nLabel Distribution:\n", label_countsKF)
    plt.figure(figsize=(5, 5))
    plt.pie(label_countsKF, labels=label_countsKF.index, autopct='%1.1f%%', startangle=90)
    plt.title('Original Distribution of Labels')
    plt.show()

In [ ]:

# Get the label distribution after training
# Get the predicted labels after training
class_counts = pd.Series(y_pred_classesKF).value_counts()
print(class_counts)
# Plot pie chart of predicted labels
plt.figure(figsize=(4, 5))
colors = ["#FF91A4", "#f6c344", "#79d970", "#F4E1C1", "#6495ED"]
plt.pie(pd.Series(y_pred_classesKF).value_counts(), labels=label_encoder.classes_, autopct='%1.1f%%', startangle=90,colors=colors)
plt.axis('equal')
plt.title('TCN-A predicted Label distribution on GPS Dataset')
plt.show()